In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
from collections.abc import Sequence

In [2]:
import pvlib.solarposition
from pvlib.solarposition import get_solarposition

In [3]:
from scipy.integrate import trapezoid

In [2]:
336*27 / 60 /60

2.52

In [4]:
def era5_tsi_data() -> xr.DataArray:
    """
    Load Total Solar Irradiance (TSI) for 1951.5–2034.5 as used in ERA5.
    The ECMWF values (IFS cycle 41r2) are scaled to match more recent observations.
    """
    TSI_SCALE = 0.9965

    # 1951–1995: non‑repeating “historical” sequence (45 values)
    tsi_hist = [
        1365.7765, 1365.7676, 1365.6284, 1365.6564, 1365.7773,
        1366.3109, 1366.6681, 1366.6328, 1366.3828, 1366.2767,
        1365.9199, 1365.7484, 1365.6963, 1365.6976, 1365.7341,
        1365.9178, 1366.1143, 1366.1644, 1366.2476, 1366.2426,
        1365.9580, 1366.0525, 1365.7991, 1365.7271, 1365.5345,
        1365.6453, 1365.8331, 1366.2747, 1366.6348, 1366.6482,
        1366.6951, 1366.2859, 1366.1992, 1365.8103, 1365.6416,
        1365.6379, 1365.7899, 1366.0826, 1366.6479, 1366.5533,
        1366.4457, 1366.3021, 1366.0286, 1365.7971, 1365.6996,
    ]

    # 1996–2008: 13‑year cycle
    tsi_cycle = [
        1365.6121, 1365.7399, 1366.1021, 1366.3851, 1366.6836,
        1366.6022, 1366.6807, 1366.2300, 1366.0480, 1365.8545,
        1365.8107, 1365.7240, 1365.6918,
    ]

    # Build full 84‑year series by repeating the 13‑year cycle for
    # 1996–2008, 2009–2021 and 2022–2034
    tsi_values = (
        tsi_hist
        + tsi_cycle                       # 1996–2008
        + tsi_cycle                       # 2009–2021
        + tsi_cycle                       # 2022–2034
    )

    # Time coordinate: 1951.5, 1952.5, …, 2034.5 (84 points)
    time = np.arange(1951.5, 2035.5, 1.0)
    
    # Apply scale and return as an xarray.DataArray
    tsi_array = TSI_SCALE * np.array(tsi_values)
    return xr.DataArray(tsi_array, dims=["time"], coords={"time": time})

In [5]:
tsi_data = era5_tsi_data()

In [23]:
# def get_tsi(timestamps: Sequence, tsi_data: xr.DataArray) -> np.array:
#     """Returns TSI values for the given timestamps.

#     TSI values are interpolated from the provided yearly TSI data.

#     Args:
#         timestamps: Timestamps for which to compute TSI values.
#         tsi_data: A DataArray with a single dimension `time` that has coordinates in
#         units of years since 0000-1-1. E.g. 2023.5 corresponds to the middle of
#         the year 2023.

#     Returns:
#         An Array containing interpolated TSI data.
#     """
#     timestamps = pd.DatetimeIndex(timestamps, tz="utc")
#     timestamps_date = pd.DatetimeIndex(timestamps.date, tz="utc")
#     day_fraction = (timestamps - timestamps_date) / pd.Timedelta(days=1)
#     year_length = 365 + timestamps.is_leap_year
#     year_fraction = (timestamps.dayofyear - 1 + day_fraction) / year_length
#     fractional_year = timestamps.year + year_fraction
    
#     return np.interp(fractional_year, tsi_data.coords["time"].data, tsi_data.data)

def get_tsi_np(
    timestamps: Sequence,
    tsi_data: xr.DataArray
) -> np.ndarray:
    """
    Pure‐NumPy interpolation of TSI for each timestamp.
    
    timestamps : array‐like of values castable to datetime64[ns]
    tsi_data   : xarray.DataArray with a 1D 'time' coord of fractional years
    """
    # 1) Cast inputs to numpy datetime64[ns]
    ts = np.asarray(timestamps, dtype="datetime64[ns]")
    
    # 2) Compute start‐of‐year (datetime64[Y]) and integer year
    start_of_year = ts.astype("datetime64[Y]")
    years = start_of_year.astype(int) + 1970
    
    # 3) Days since start‐of‐year
    days_since_start = (ts - start_of_year) / np.timedelta64(1, "D")
    
    # 4) Leap‐year test → 365 or 366 days
    is_leap = ((years % 4 == 0) &
               ((years % 100 != 0) | (years % 400 == 0)))
    days_in_year = np.where(is_leap, 366, 365)
    
    # 5) Fractional year
    frac_year = years + days_since_start / days_in_year

    # 6) Extract TSI arrays and cast to float
    time_vals = tsi_data.coords["time"].values.astype(float)
    tsi_vals  = tsi_data.values.astype(float)
    
    # 7) Vectorized interpolation
    return np.interp(frac_year, time_vals, tsi_vals)

In [33]:
# def get_toa_radiation(start_date: str, end_date: str, step_freq: str = "1h", sub_freq: str = "10Min"):
#     """
#     Calculate top of atmosphere solar irradiance

#     Args:
#         start_date (str): Start date of time series
#         end_date (str): End date of time series (inclusive).
#         step_freq (str): How much time between steps in pandas time string format (e.g., 1h, 10Min)
#         sub_freq (str): How much time between substeps that are integrated forward (e.g., 10Min)

#     Returns:
#         top of atmosphere radiation in W m**-2.
#     """
#     start_date_ts = pd.Timestamp(start_date)
#     end_date_ts = pd.Timestamp(end_date)
#     dates = pd.date_range(
#         start=start_date_ts - pd.Timedelta(step_freq) + pd.Timedelta(sub_freq),
#         end=end_date_ts,
#         freq=sub_freq,
#         tz="utc",
#     )
    
#     total_rad = get_tsi_np(dates, tsi_data)
#     solar_distance = pvlib.solarposition.nrel_earthsun_distance(dates, how="numba")
#     solar_factor = (1.0 / solar_distance) ** 2
    
#     return total_rad * solar_factor

import re

# helper to parse strings like "1h", "10Min", "2D" → np.timedelta64
_unit_map = {
    "h": "h",
    "hour": "h", "Hour": "h", "H": "h",
    "min": "m",  "Min": "m",  "M": "m",
    "d": "D",    "D": "D",
}

def _parse_freq(freq: str) -> np.timedelta64:
    m = re.fullmatch(r"(\d+)\s*([A-Za-z]+)", freq)
    if not m:
        raise ValueError(f"Could not parse freq '{freq}'")
    n, unit = int(m.group(1)), m.group(2)
    u = _unit_map.get(unit)
    if u is None:
        raise ValueError(f"Unrecognized unit '{unit}' in freq '{freq}'")
    return np.timedelta64(n, u)

def get_toa_radiation(
    start_date: str,
    end_date:   str,
    step_freq:  str = "1h",
    sub_freq:   str = "10Min",
) -> np.ndarray:
    """
    Calculate TOA solar irradiance at `sub_freq` resolution,
    using pure‑NumPy date arrays and your get_tsi_np function.
    """
    # 1) Parse inputs as numpy datetimes/timedeltas
    start = np.datetime64(start_date, "ns")
    end   = np.datetime64(end_date,   "ns")
    step  = _parse_freq(step_freq)
    sub   = _parse_freq(sub_freq)

    # 2) Build the substep timeline once, in np.datetime64
    #    start - step + sub  → end (inclusive of end)
    dates = np.arange(start - step + sub, end + sub, sub, dtype="datetime64[ns]")

    # 3) Fetch TSI for each timepoint (pure NumPy)
    total_rad = get_tsi_np(dates, tsi_data)

    # 4) Compute Earth–Sun distance factor.
    # pvlib’s numba version accepts numpy datetime64[ns] in recent releases.
    try:
        dist = pvlib.solarposition.nrel_earthsun_distance(dates, how="numba")
    except TypeError:
        # fallback to pandas index only if needed
        import pandas as pd
        idx = pd.DatetimeIndex(dates, tz="utc")
        dist = pvlib.solarposition.nrel_earthsun_distance(idx, how="numba")

    # 5) Return the TOA irradiance
    return total_rad * (1.0 / dist) ** 2

In [38]:
# _unit_map = {
#     "h": "h", "hour": "h", "H": "h",
#     "min": "m", "Min": "m", "M": "m",
#     "d": "D", "D": "D",
# }
# def _parse_freq(freq: str) -> np.timedelta64:
#     m = re.fullmatch(r"(\d+)\s*([A-Za-z]+)", freq)
#     if not m:
#         raise ValueError(f"Could not parse freq '{freq}'")
#     n, u = int(m.group(1)), m.group(2)
#     unit = _unit_map.get(u)
#     if unit is None:
#         raise ValueError(f"Unrecognized unit '{u}' in freq '{freq}'")
#     return np.timedelta64(n, unit)

def get_solar_radiation_loc(
    toa_radiation,           # array‑like of TOA (W m⁻²) at each substep
    lon: float,
    lat: float,
    altitude: float,
    start_date: str,
    end_date:   str,
    step_freq:  str = "1h",
    sub_freq:   str = "10Min",
) -> xr.Dataset:
    
    # 1) parse dates and freqs
    start = np.datetime64(start_date, "ns")
    end   = np.datetime64(end_date,   "ns")
    step  = _parse_freq(step_freq)
    sub   = _parse_freq(sub_freq)

    # 2) build substep times: from (start-step+sub) to ≤ end, every sub
    dates_sub = np.arange(start - step + sub, end + sub, sub, dtype="datetime64[ns]")

    # 3) ensure toa_radiation lines up
    toa = np.asarray(toa_radiation, float)
    if toa.shape[0] != dates_sub.shape[0]:
        raise ValueError(f"Length mismatch: toa={toa.shape[0]}, dates={dates_sub.shape[0]}")

    # 4) get solar zenith (vectorized). fallback to pandas only if needed
    try:
        solpos = get_solarposition(dates_sub, lat, lon, altitude, method="nrel_numba")
    except TypeError:
        import pandas as pd
        idx = pd.DatetimeIndex(dates_sub, tz="utc")
        solpos = get_solarposition(idx, lat, lon, altitude, method="nrel_numba")

    zen = solpos["zenith"].values
    cos_zen = np.maximum(0, np.cos(np.radians(zen)))

    # 5) compute down‑welling rad at each substep
    rad_sub = toa * cos_zen

    # 6) reshape into (n_steps, step_len)
    step_len = int(step // sub)
    n_steps  = rad_sub.size // step_len
    grid     = rad_sub.reshape(n_steps, step_len)

    # 7) integrate with NumPy’s trapz (dx in seconds)
    dx_seconds = float(sub / np.timedelta64(1, "s"))
    step_rad   = np.trapz(grid, dx=dx_seconds, axis=1)

    # 8) pick cos_zen at the last substep of each block
    cos_grid  = cos_zen.reshape(n_steps, step_len)
    step_cos  = cos_grid[:, -1]

    # 9) build step timestamps (naive UTC datetime64[ns])
    step_times = start + np.arange(n_steps) * step

    # 10) pack into xarray.Dataset
    ds = xr.Dataset(
        {
            "tsi":   (("time","latitude","longitude"), step_rad[:,None,None]),
            "coszen":(("time","latitude","longitude"), step_cos[:,None,None]),
        },
        coords={
            "time":      step_times,
            "latitude":  [lat],
            "longitude": [lon],
        },
    )
    ds["tsi"].attrs = {
        "standard_name": "solar_irradiance",
        "long_name":     "total solar irradiance",
        "units":         "J m-2",
    }
    ds["coszen"].attrs = {
        "standard_name": "cos_solar_zenith_angle",
        "long_name":     "cosine of the solar zenith angle",
        "units":         "",
    }
    return ds

# def get_solar_radiation_loc(
#     toa_radiation: pd.Series,
#     lon: float,
#     lat: float,
#     altitude: float,
#     start_date: str,
#     end_date: str,
#     step_freq: str = "1h",
#     sub_freq: str = "10Min",
# ) -> xr.Dataset:
#     """
#     Calculate total solar irradiance at a single location over a range of times. Solar irradiance is integrated
#     over the step frequency at specified substeps.

#     Args:
#         toa_radiation (pd.Series): Top of atmosphere solar radiation in W m**-2.
#         lon (float): longitude.
#         lat (float): latitude.
#         altitude (float): altitude in meters.
#         start_date (str): date str for the beginning of the period (inclusive).
#         end_date (str):  date str for the end of the period (inclusive).
#         step_freq (str): period over which irradiance is integrated. Defaults to "1h".
#         sub_freq (str): sub step frequency over the step period. Defaults to "5Min".

#     Returns:
#         xarray.Dataset: total solar irradiance and cosine of solar zenith angle time series with metadata.
#     """
#     start_date_ts = pd.Timestamp(start_date)
#     end_date_ts = pd.Timestamp(end_date)
#     step_sec = pd.Timedelta(step_freq).total_seconds()
#     sub_sec = pd.Timedelta(sub_freq).total_seconds()
#     step_len = int(step_sec // sub_sec)
#     dates = pd.date_range(
#         start=start_date_ts - pd.Timedelta(step_freq) + pd.Timedelta(sub_freq),
#         end=end_date_ts,
#         freq=sub_freq,
#         tz="utc",
#     )
#     solar_pos = get_solarposition(dates, lat, lon, altitude, method="nrel_numba")
#     cos_zenith = np.maximum(0, np.cos(np.radians(solar_pos["zenith"].values)))
#     solar_rad = toa_radiation * cos_zenith

#     step_rad = trapezoid(
#         np.reshape(solar_rad, (int(solar_rad.size // step_len), step_len)),
#         dx=sub_sec,
#         axis=1,
#     )
#     step_dates = pd.date_range(start=start_date_ts, end=end_date_ts, freq=step_freq, tz="utc")
#     step_cos_zenith = pd.Series(cos_zenith, index=dates)[step_dates]

#     out_rad_da = xr.DataArray(
#         step_rad.reshape(-1, 1, 1),
#         coords={"time": step_dates, "latitude": [lat], "longitude": [lon]},
#         dims=("time", "latitude", "longitude"),
#         name="tsi",
#         attrs={
#             "standard_name": "solar_irradiance",
#             "long_name": "total solar irradiance",
#             "units": "J m-2",
#         },
#     )

#     zenith_da = xr.DataArray(
#         step_cos_zenith.values.reshape(-1, 1, 1),
#         coords={"time": step_dates, "latitude": [lat], "longitude": [lon]},
#         dims=("time", "latitude", "longitude"),
#         name="solar_zenith_angle",
#         attrs={
#             "standard_name": "cos_solar_zenith_angle",
#             "long_name": "cosine of the solar zenith angle",
#             "units": "",
#         },
#     )
#     out_rad_ds = xr.Dataset({"tsi": out_rad_da, "coszen": zenith_da})
#     return out_rad_ds


# # def get_solar_index(curr_date, ref_date="2000-01-01"):
# #     curr_date_ts = pd.to_datetime(curr_date)
# #     year_start = pd.Timestamp(f"{curr_date_ts.year:d}-01-01")
# #     curr_diff = curr_date_ts - year_start
# #     return int(curr_diff.total_seconds() / 3600)

In [39]:

lons = np.arange(-100.0, -89.5, 0.5)
lats = np.arange(30.0, 35.0, 0.5)
lon_grid, lat_grid = np.meshgrid(lons, lats)
solar_ts = []
toa_radiation = get_toa_radiation("2016-01-01", "2016-12-31 23:00")
for lon_val, lat_val in tqdm(zip(lon_grid.ravel(), lat_grid.ravel())):
    out = get_solar_radiation_loc(toa_radiation, lon_val, lat_val, 0.0, "2016-01-01", "2016-12-31 23:00")
    solar_ts.append(out)
combined = xr.combine_by_coords(solar_ts)
print(combined['tsi'].isel(time=999).values[:, 7])

210it [00:16, 13.09it/s]


[1177682.32320883 1161424.88949503 1145078.99941375 1128645.89785644
 1112126.83635525 1095523.07298773 1078835.87228101 1062066.50511541
 1045216.24862773 1028286.38611386]


```python
array([1177682.32320883, 1161424.88949503, 1145078.99941375,
       1128645.89785644, 1112126.83635525, 1095523.07298773,
       1078835.87228101, 1062066.50511541, 1045216.24862773,
       1028286.38611386])
```